# GrandQC × IDC Pipeline v4

WSI quality control pipeline using GrandQC on NCI Imaging Data Commons slides.

**GitHub:** `github.com/ronsong1234/grandqc-idc-pipeline`

| Cell | Purpose |
|------|---------|
| 1 | Install system + Python dependencies |
| 2 | Clone GrandQC + download 7× weights (cached) |
| 3 | Patch GrandQC for IDC TIFF compatibility |
| 4 | CONFIG — collections, n_slides, magnification |
| 5 | Download DICOM from IDC (cached) |
| 6 | Convert DICOM → TIFF (memory-safe, auto-downsample) |
| 7 | Run GrandQC — tissue detection + artifact segmentation at 7× |
| 8 | Compute QC metrics → qc_summary_v4.csv |
| 9a | Install + patch HistoQC for Python 3.12 |
| 9b | Run HistoQC on all TIFFs |
| 9c | GrandQC vs HistoQC comparison table |
| 10 | Clean GeoJSONs + package QuPath zip |

**Key decisions:**
- Magnification: 7× (MPP 1.5) — GrandQC paper benchmark, Dice 0.808
- Metrics: % of detected tissue area (not total slide)
- Pass/fail: PASS ≥80% · BORDERLINE 50–80% · FAIL <50% no-artifact
- Large slides (canvas >4GB) auto-downsampled to 1.0 MPP


In [ ]:
# ── Cell 1 — Install dependencies ────────────────────────────────────────────
import subprocess, sys

subprocess.run(['apt-get', 'install', '-y', '-q',
                'openslide-tools', 'libglib2.0-0'], check=True)

packages = [
    'openslide-python', 'pydicom', 'tifffile', 'idc-index',
    'opencv-python-headless', 'scikit-image', 'tqdm'
]

for pkg in packages:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True)
    status = '✓' if r.returncode == 0 else '[warn]'
    print(f'  {status} {pkg}')

print('\n✓ Cell 1 complete')


  ✓ openslide-python
  ✓ pydicom
  ✓ tifffile
  ✓ idc-index
  ✓ opencv-python-headless
  ✓ scikit-image
  ✓ tqdm

✓ Cell 1 complete


In [ ]:
# ── Cell 2 — Clone GrandQC + download 7× weights (cached) ───────────────────
import os, sys, subprocess, urllib.request
from pathlib import Path
from tqdm.notebook import tqdm

WORK_DIR     = Path('/content/grandqc_idc')
PIPELINE_DIR = WORK_DIR / 'grandqc' / '01_WSI_inference_OPENSLIDE_QC'
TIFF_DIR     = WORK_DIR / 'tiff'
QC_BASE      = WORK_DIR / 'grandqc_output'
WORK_DIR.mkdir(exist_ok=True)
os.chdir(WORK_DIR)

# Clone repo
if not (WORK_DIR / 'grandqc').exists():
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/cpath-ukk/grandqc.git'], check=True)
    print('✓ GrandQC repo cloned')
else:
    print('✓ GrandQC repo already present')

# Install requirements (skip torch/numpy — already in Colab)
skip = ('torch', 'torchvision', 'torchaudio', 'numpy')
reqs_path = WORK_DIR / 'grandqc' / 'requirements.txt'
reqs = [
    r for r in reqs_path.read_text().splitlines()
    if r.strip() and not r.startswith('#')
    and not any(r.lower().startswith(s) for s in skip)
]
import importlib
to_install = []
for req in reqs:
    pkg = req.split('==')[0].split('>=')[0].split('<=')[0].strip().replace('-','_').lower()
    try:
        importlib.import_module(pkg)
    except ImportError:
        to_install.append(req)

if to_install:
    print(f'Installing {len(to_install)} missing packages...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'timm==0.4.12', 'pretrainedmodels',
        'efficientnet-pytorch', 'segmentation-models-pytorch'], check=True)
    print('✓ Packages installed')
else:
    print('✓ All packages already installed')

# Download 7× weights only (paper benchmark — skip 5× and 10×)
TD  = PIPELINE_DIR / 'models' / 'td'
QCM = PIPELINE_DIR / 'models' / 'qc'
TD.mkdir(parents=True, exist_ok=True)
QCM.mkdir(parents=True, exist_ok=True)

WEIGHTS = [
    ('https://zenodo.org/records/14507273/files/Tissue_Detection_MPP10.pth',
     TD  / 'Tissue_Detection_MPP10.pth'),
    ('https://zenodo.org/records/14041538/files/GrandQC_MPP15.pth',
     QCM / 'GrandQC_MPP15.pth'),
]

class Bar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize: self.total = tsize
        self.update(b * bsize - self.n)

all_cached = True
for url, dest in WEIGHTS:
    if dest.exists() and dest.stat().st_size > 1e6:
        print(f'  cached  {dest.name}  ({dest.stat().st_size/1e6:.0f} MB)')
    else:
        all_cached = False
        print(f'  downloading {dest.name}...')
        with Bar(unit='B', unit_scale=True, desc=dest.name) as t:
            urllib.request.urlretrieve(url, dest, reporthook=t.update_to)
        print(f'  ✓ {dest.name}  ({dest.stat().st_size/1e6:.0f} MB)')

if all_cached:
    print('\n✓ All weights cached — Cell 2 ran in seconds')
else:
    print('\n✓ Cell 2 complete — 7× weights ready')


✓ GrandQC repo already present
Installing 5 missing packages...
✓ Packages installed
  cached  Tissue_Detection_MPP10.pth  (27 MB)
  cached  GrandQC_MPP15.pth  (25 MB)

✓ All weights cached — Cell 2 ran in seconds


In [ ]:
# ── Cell 3 — Patch GrandQC source files for IDC TIFF compatibility ────────────
# All patches are idempotent — safe to re-run after session restarts.
# Run this cell ONCE after Cell 2 (or any time you re-clone GrandQC).
import re, shutil
from pathlib import Path

PIPELINE_DIR = Path('/content/grandqc_idc/grandqc/01_WSI_inference_OPENSLIDE_QC')

# ── Patch 1: wsi_slide_info.py — MPP fallback for IDC TIFFs ──────────────────
# OpenSlide cannot read MPP from deflate-compressed TIFFs written by Cell 6.
# Falls back to tifffile XResolution tag (written as px/cm by Cell 6).
f = PIPELINE_DIR / 'wsi_slide_info.py'
content = f.read_text()

old_mpp = '    # Microne per pixel\n    mpp = float(slide.properties["openslide.mpp-x"])\n    p_s = int(mpp_model / mpp * m_p_s)'
new_mpp = '''    # Microns per pixel — tifffile fallback for IDC deflate TIFFs
    try:
        mpp = float(slide.properties["openslide.mpp-x"])
    except Exception:
        mpp = None
    if mpp is None:
        try:
            import tifffile as _tf
            with _tf.TiffFile(str(slide.filename)) as _tif:
                _page = _tif.pages[0]
                _xres  = _page.tags.get('XResolution')
                _runit = _page.tags.get('ResolutionUnit')
                if _xres is not None:
                    _val = _xres.value
                    if isinstance(_val, tuple):
                        _val = _val[0] / _val[1] if _val[1] != 0 else _val[0]
                    _runit_val = getattr(_runit, 'value', 2) if _runit else 2
                    if _runit_val == 3:    # centimeter
                        mpp = round(10000.0 / float(_val), 4)
                    elif _runit_val == 2:  # inch
                        mpp = round(25400.0 / float(_val), 4)
        except Exception:
            pass
    if mpp is None:
        mpp = 1.0  # final fallback
    p_s = int(mpp_model / mpp * m_p_s)'''

if 'tifffile fallback' not in content:
    if old_mpp in content:
        content = content.replace(old_mpp, new_mpp)
        f.write_text(content)
        print('✓ wsi_slide_info.py — MPP tifffile fallback patched')
    else:
        print('[warn] wsi_slide_info.py — old string not found; showing lines 13-22:')
        for i, ln in enumerate(content.splitlines()[12:22], 13):
            print(f'  {i:4}: {repr(ln)}')
else:
    print('[cached] wsi_slide_info.py')

# ── Patch 2: wsi_tis_detect.py — MPP fallback (line 82) + MIN_TD=512 + torch fix ──
# Line 82 is a separate MPP read from wsi_slide_info.py — needs its own fallback.
# Uses path_slide (string) not slide._filename (doesn't exist in tissue detector).
# MIN_TD=512 prevents tissue=100% bug on small IDC slides.
# torch.tensor copy required for PyTorch 2.x.
f = PIPELINE_DIR / 'wsi_tis_detect.py'
content = f.read_text()
changed = False

if 'tifffile fallback line82' not in content:
    old82 = '        mpp = round(float(slide.properties["openslide.mpp-x"]), 4)'
    new82 = '''        # tifffile fallback line82
        try:
            mpp = round(float(slide.properties["openslide.mpp-x"]), 4)
        except Exception:
            mpp = None
        if mpp is None:
            try:
                import tifffile as _tf
                with _tf.TiffFile(path_slide) as _tif:
                    _page = _tif.pages[0]
                    _xres  = _page.tags.get('XResolution')
                    _runit = _page.tags.get('ResolutionUnit')
                    if _xres is not None:
                        _val = _xres.value
                        if isinstance(_val, tuple):
                            _val = _val[0] / _val[1] if _val[1] != 0 else _val[0]
                        _runit_val = getattr(_runit, 'value', 2) if _runit else 2
                        if _runit_val == 3:
                            mpp = round(10000.0 / float(_val), 4)
                        elif _runit_val == 2:
                            mpp = round(25400.0 / float(_val), 4)
            except Exception:
                pass
        if mpp is None:
            mpp = 1.0'''
    if old82 in content:
        content = content.replace(old82, new82)
        changed = True
    else:
        print('[warn] wsi_tis_detect.py — line 82 original not found; showing lines 78-90:')
        for i, ln in enumerate(content.splitlines()[77:90], 78):
            print(f'  {i:4}: {repr(ln)}')

if 'MIN_TD = 512' not in content:
    content = content.replace('MIN_TD = 256', 'MIN_TD = 512')
    changed = True

if 'torch.tensor(image_pre.copy())' not in content:
    content = content.replace('torch.tensor(image_pre)', 'torch.tensor(image_pre.copy())')
    changed = True

f.write_text(content)
if changed:
    print('✓ wsi_tis_detect.py — MPP fallback (line 82) + MIN_TD=512 + torch.tensor patched')
else:
    print('[cached] wsi_tis_detect.py')

# ── Patch 3: main.py — weights_only=False for PyTorch 2.x ────────────────────
f = PIPELINE_DIR / 'main.py'
content = f.read_text()
if 'weights_only=False' not in content:
    content = content.replace(
        'torch.load(MODEL_QC_DIR + MODEL_QC_NAME, map_location=DEVICE)',
        'torch.load(MODEL_QC_DIR + MODEL_QC_NAME, map_location=DEVICE, weights_only=False)'
    ).replace(
        'torch.load(MODEL_TD_DIR + MODEL_TD_NAME, map_location=DEVICE)',
        'torch.load(MODEL_TD_DIR + MODEL_TD_NAME, map_location=DEVICE, weights_only=False)'
    )
    f.write_text(content)
    print('✓ main.py — weights_only=False patched')
else:
    print('[cached] main.py')

# ── Patch 4: wsi_process.py — tissue mask convention ─────────────────────────
# IDC TIFFs: tissue=black=0, background=white=1 (opposite of GrandQC default).
# Threshold fix: mask values are 0-255 integers, not 0-1 floats.
f = PIPELINE_DIR / 'wsi_process.py'
content = f.read_text()
if 'IDC tissue convention' not in content:
    old4 = 'mask = np.where(td_patch_ == 0, BACK_CLASS, mask_raw)'
    new4 = ('# IDC tissue convention: tissue=black=0, background=white=1\n'
            '            mask = np.where(td_patch_ == 0, BACK_CLASS, mask_raw)')
    if old4 in content:
        content = content.replace(old4, new4)
        content = content.replace('> 0.5', '> 50')
        f.write_text(content)
        print('✓ wsi_process.py — tissue mask convention + threshold patched')
    else:
        # Regex fallback
        import re as _re
        new_content, n = _re.subn(
            r'([ \t]*)mask = np\.where\(td_patch_ == 0, BACK_CLASS, mask_raw\)',
            lambda m: (m.group(1) + '# IDC tissue convention: tissue=black=0, background=white=1\n'
                       + m.group(1) + 'mask = np.where(td_patch_ == 1, BACK_CLASS, mask_raw)'),
            content
        )
        if n > 0:
            new_content = new_content.replace('> 0.5', '> 50')
            f.write_text(new_content)
            print('✓ wsi_process.py — tissue mask convention patched (regex fallback)')
        else:
            lines = content.splitlines()
            matches = [(i+1, ln) for i, ln in enumerate(lines)
                       if 'td_patch_' in ln or 'BACK_CLASS' in ln]
            print('[warn] wsi_process.py — could not auto-patch. Relevant lines:')
            for lineno, ln in matches:
                print(f'  {lineno:4}: {ln}')
else:
    print('[cached] wsi_process.py')

# ── Clear pycache so all patches take effect ──────────────────────────────────
for d in PIPELINE_DIR.rglob('__pycache__'):
    shutil.rmtree(d, ignore_errors=True)
print('✓ pycache cleared')

print('\n✓ Cell 3 complete — all 4 patches applied')
print('   Re-run this cell any time after a session restart or re-clone')


[cached] wsi_slide_info.py
✓ wsi_tis_detect.py — MPP fallback (line 82) + MIN_TD=512 + torch.tensor patched
[cached] main.py
[warn] wsi_process.py — could not auto-patch. Relevant lines:
    39:                          ENCODER_MODEL_1,ENCODER_WEIGHTS, DEVICE, BACK_CLASS, MPP_MODEL_1, mpp, w_l0, h_l0):
    72:                 td_patch_ = np.pad(td_patch, padding, mode='constant')
    74:                 td_patch_ = td_patch
    90:                 mask = np.where(td_patch_ == 1, BACK_CLASS, mask_raw)
    94:                 mask = np.full((512,512), BACK_CLASS)
✓ pycache cleared

✓ Cell 3 complete — all 4 patches applied
   Re-run this cell any time after a session restart or re-clone


In [ ]:
# ── Cell 4 — CONFIG ────────────────────────────────────────────────────────────
from pathlib import Path
from idc_index import IDCClient

# ── Settings — edit these ─────────────────────────────────────────────────────
COLLECTIONS = {
    'tcga_brca': 'breast',       # change collection as needed
}

N_SLIDES       = 1        # slides per collection (1 recommended for large TCGA)
MPP_FILTER     = 3.0      # exclude slides with MPP > this (not diagnostic WSIs)
MAGNIFICATIONS = [1.5]    # 7× only — GrandQC paper benchmark (Dice 0.808)

# ── Paths (auto-set, do not change) ───────────────────────────────────────────
WORK_DIR     = Path('/content/grandqc_idc')
PIPELINE_DIR = WORK_DIR / 'grandqc' / '01_WSI_inference_OPENSLIDE_QC'
TIFF_DIR     = WORK_DIR / 'tiff'
QC_BASE      = WORK_DIR / 'grandqc_output'

# ── Query IDC ──────────────────────────────────────────────────────────────────
client = IDCClient()
print(f'IDC {client.get_idc_version()}')
client.fetch_index('sm_index')

col_list = ', '.join(f"'{c}'" for c in COLLECTIONS)
df = client.sql_query(f"""
    SELECT i.collection_id, i.PatientID, i.StudyInstanceUID,
           i.SeriesInstanceUID, ROUND(i.series_size_MB, 1) AS size_MB,
           i.license_short_name, s.ObjectiveLensPower AS lens_power,
           s.max_TotalPixelMatrixColumns AS width,
           s.max_TotalPixelMatrixRows    AS height
    FROM index i
    JOIN sm_index s ON i.SeriesInstanceUID = s.SeriesInstanceUID
    WHERE i.Modality = 'SM'
    AND i.collection_id IN ({col_list})
    AND i.series_size_MB > 100
    AND s.max_TotalPixelMatrixColumns > 50000
    AND s.max_TotalPixelMatrixRows    > 50000
    ORDER BY i.collection_id, i.series_size_MB DESC
""")
df['cancer_label'] = df['collection_id'].map(COLLECTIONS)

sampled = (
    df.groupby('collection_id', group_keys=False)
      .head(N_SLIDES)
      .reset_index(drop=True)
)

print(f'\nSelected {len(sampled)} slides:')
print(sampled[['cancer_label','collection_id','PatientID',
               'size_MB','width','height','lens_power']].to_string(index=False))
print(f'\nMagnification: 7x (MPP 1.5) — GrandQC paper benchmark')


IDC v24

Selected 1 slides:
cancer_label collection_id    PatientID  size_MB  width  height  lens_power
      breast     tcga_brca TCGA-OL-A66K   3586.3 139008  256256          40

Magnification: 7x (MPP 1.5) — GrandQC paper benchmark


In [ ]:
# ── Cell 5 — Download DICOM from IDC (cached) ─────────────────────────────────
from pathlib import Path
from idc_index import IDCClient

WORK_DIR  = Path('/content/grandqc_idc')
DICOM_DIR = WORK_DIR / 'dicom'
DICOM_DIR.mkdir(exist_ok=True)

client = IDCClient()
series_dirs = {}

for _, row in sampled.iterrows():
    uid  = row['SeriesInstanceUID']
    sdir = (DICOM_DIR / row['collection_id'] / row['PatientID']
            / row['StudyInstanceUID'] / uid)

    if sdir.exists() and any(sdir.glob('*.dcm')):
        n  = len(list(sdir.glob('*.dcm')))
        sz = sum(f.stat().st_size for f in sdir.glob('*.dcm')) / 1e6
        print(f'  [cached]   {row["collection_id"]}  {row["PatientID"]}  ({n} files, {sz:.0f} MB)')
        series_dirs[uid] = sdir
        continue

    print(f'  [download] {row["collection_id"]}  {row["PatientID"]}  ({row["size_MB"]:.0f} MB)')
    print(f'             Large TCGA slides may take 20-40 minutes...')
    client.download_from_selection(
        seriesInstanceUID=[uid],
        downloadDir=str(DICOM_DIR),
        dirTemplate='%collection_id/%PatientID/%StudyInstanceUID/%SeriesInstanceUID'
    )
    series_dirs[uid] = sdir
    print(f'  ✓ Downloaded')

print(f'\n✓ {len(series_dirs)} series ready')

  [download] tcga_brca  TCGA-OL-A66K  (3586 MB)
             Large TCGA slides may take 20-40 minutes...


  ✓ Downloaded

✓ 1 series ready


In [ ]:
# ── Cell 6 — Convert DICOM → TIFF (wsidicom streaming) ───────────────────────
import subprocess, sys, math, gc
import numpy as np
import tifffile
from pathlib import Path

from pathlib import Path
TIFF_DIR = Path('/content/grandqc_idc/tiff')
for f in TIFF_DIR.glob('*.tiff'):
    f.unlink()
    print(f'deleted {f.name}')

# ── Install required packages ─────────────────────────────────────────────────
subprocess.run(['apt-get', 'install', '-y', '-q', 'libvips-tools'], check=True)
for pkg in ['wsidicom', 'imagecodecs', 'pyvips']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                   capture_output=True)

# ── Safety: redefine paths ────────────────────────────────────────────────────
WORK_DIR     = Path('/content/grandqc_idc')
PIPELINE_DIR = WORK_DIR / 'grandqc' / '01_WSI_inference_OPENSLIDE_QC'
TIFF_DIR     = WORK_DIR / 'tiff'
QC_BASE      = WORK_DIR / 'grandqc_output'
MPP_FILTER   = 3.0
TIFF_DIR.mkdir(exist_ok=True)

import torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('✓ Memory cleared')

# ── Delete any empty/broken TIFFs from previous runs ─────────────────────────
for f in TIFF_DIR.glob('*.tiff'):
    if f.stat().st_size < 10_000:
        f.unlink()
        print(f'  deleted broken: {f.name}')

def dicom_to_tiff_native(dcm_dir, out_path, target_mpp=1.0):
    """
    Streaming DICOM → TIFF using wsidicom.
    - Reads one 512×512 tile at a time — never loads full slide into RAM
    - Auto-downsamples to target_mpp if canvas > 2GB
    - Writes pyramidal TIFF via pyvips so OpenSlide get_thumbnail() works
    """
    from wsidicom import WsiDicom
    from PIL import Image as PILImg
    import numpy as np, math, gc
    from pathlib import Path

    dcm_dir = Path(dcm_dir)
    print(f'    opening with wsidicom...')

    with WsiDicom.open(str(dcm_dir)) as wsi:
        level     = wsi.levels.base_level
        W         = level.size.width
        H         = level.size.height
        mpp_val   = level.mpp.width if level.mpp is not None else None
        mpp       = float(mpp_val) if mpp_val else 0.2425
        if mpp_val is None:
            print(f'    [warn] MPP not found, defaulting to {mpp}')

        canvas_gb = W * H * 3 / 1e9
        print(f'    native {W}×{H} px  mpp={mpp:.4f}  canvas={canvas_gb:.1f} GB')

        # Downsample if canvas > 2GB
        if canvas_gb > 2 and mpp < target_mpp:
            scale   = target_mpp / mpp
            new_W   = int(W / scale)
            new_H   = int(H / scale)
            out_mpp = target_mpp
            print(f'    downsampling {scale:.1f}x → {new_W}×{new_H} px ({new_W*new_H*3/1e9:.2f} GB)')
        else:
            scale   = 1.0
            new_W, new_H = W, H
            out_mpp = mpp

        canvas = np.full((new_H, new_W, 3), 255, dtype=np.uint8)

        # Stream tiles
        TILE        = 512
        total_tiles = math.ceil(H / TILE) * math.ceil(W / TILE)
        tile_count  = 0

        for y in range(0, H, TILE):
            for x in range(0, W, TILE):
                x2 = min(x + TILE, W)
                y2 = min(y + TILE, H)
                try:
                    region = wsi.read_region((x, y), 0, (x2-x, y2-y))
                    patch = np.array(region)[:, :, :3]
                except Exception:
                    tile_count += 1
                    continue

                if scale > 1:
                    patch = np.array(
                        PILImg.fromarray(patch).resize(
                            (max(1, int(patch.shape[1]/scale)),
                             max(1, int(patch.shape[0]/scale))),
                            PILImg.LANCZOS))
                    ny  = int(y / scale)
                    nx  = int(x / scale)
                    ny2 = min(ny + patch.shape[0], new_H)
                    nx2 = min(nx + patch.shape[1], new_W)
                    canvas[ny:ny2, nx:nx2] = patch[:ny2-ny, :nx2-nx]
                else:
                    canvas[y:y2, x:x2] = patch

                tile_count += 1
                if tile_count % 200 == 0:
                    pct = tile_count / total_tiles * 100
                    print(f'    tile {tile_count}/{total_tiles} ({pct:.0f}%)')
                del patch

        # Write pyramidal TIFF via pyvips — JPEG compressed so OpenSlide
        # can decode tiles for get_thumbnail() (deflate pyramids return white).
        print(f'    canvas check — mean: {canvas.mean():.1f}  min: {canvas.min()}  max: {canvas.max()}')
        print(f'    writing pyramidal TIFF (pyvips, {out_mpp:.4f} MPP)...')
        out_path.parent.mkdir(parents=True, exist_ok=True)
        import pyvips
        height, width, bands = canvas.shape
        vips_img = pyvips.Image.new_from_memory(
            canvas.tobytes(), width, height, bands, 'uchar'
        )
        px_per_mm = 1000.0 / out_mpp
        vips_img = vips_img.copy(xres=px_per_mm, yres=px_per_mm)
        vips_img.tiffsave(
            str(out_path),
            tile=True,
            tile_width=256,
            tile_height=256,
            pyramid=True,
            compression='jpeg',
            Q=85,
            bigtiff=True,
        )

        del canvas
        gc.collect()
        return out_mpp

# ── Convert all slides ─────────────────────────────────────────────────────────
manifest = []
excluded = []

for _, row in sampled.iterrows():
    uid     = row['SeriesInstanceUID']
    dcm_dir = series_dirs.get(uid)
    if not dcm_dir or not dcm_dir.exists():
        print(f'  [skip] DICOM missing — re-run Cell 5')
        continue

    name = f"{row['cancer_label']}__{row['collection_id']}__{uid[-16:]}.tiff"
    path = TIFF_DIR / name

    if path.exists() and path.stat().st_size > 10_000:
        print(f'  [cached] {name}  ({path.stat().st_size/1e6:.1f} MB)')
        manifest.append({
            'tiff_name': name, 'tiff_path': str(path),
            'cancer_label': row['cancer_label'],
            'collection_id': row['collection_id'],
            'PatientID': row['PatientID'],
            'SeriesInstanceUID': uid,
            'mpp': None, 'size_MB': row['size_MB'],
            'lens_power': row.get('lens_power'),
            'license': row['license_short_name'],
        })
        continue

    print(f'\n  [convert] {name}')
    try:
        mpp = dicom_to_tiff_native(dcm_dir, path)
    except Exception as e:
        import traceback; traceback.print_exc()
        continue

    if mpp > MPP_FILTER:
        print(f'    ⚠️  MPP {mpp:.4f} > {MPP_FILTER} — EXCLUDED')
        excluded.append(name)
        if path.exists(): path.unlink()
        continue

    size_mb = path.stat().st_size / 1e6
    print(f'    → {size_mb:.1f} MB  mpp={mpp:.4f} ✓')

    # Verify OpenSlide can read it and MPP is present
    try:
        import openslide
        slide = openslide.OpenSlide(str(path))
        mpp_check = slide.properties.get(openslide.PROPERTY_NAME_MPP_X, 'NOT FOUND')
        print(f'    OpenSlide MPP: {mpp_check}')
        slide.close()
    except Exception as e:
        print(f'    [warn] OpenSlide check failed: {e}')

    manifest.append({
        'tiff_name': name, 'tiff_path': str(path),
        'cancer_label': row['cancer_label'],
        'collection_id': row['collection_id'],
        'PatientID': row['PatientID'],
        'SeriesInstanceUID': uid,
        'mpp': round(mpp, 4), 'size_MB': row['size_MB'],
        'lens_power': row.get('lens_power'),
        'license': row['license_short_name'],
    })

print(f'\n✓ {len(manifest)} TIFFs ready')
if excluded:
    print(f'⚠️  {len(excluded)} excluded (MPP > {MPP_FILTER}):')
    for e in excluded: print(f'   {e}')

✓ Memory cleared

  [convert] breast__tcga_brca__637595675116.2.0.tiff


    opening with wsidicom...
    native 139008×256256 px  mpp=0.1644  canvas=106.9 GB
    downsampling 6.1x → 22848×42120 px (2.89 GB)
    tile 200/136272 (0%)
    tile 400/136272 (0%)
    tile 600/136272 (0%)
    tile 800/136272 (1%)
    tile 1000/136272 (1%)
    tile 1200/136272 (1%)
    tile 1400/136272 (1%)
    tile 1600/136272 (1%)
    tile 1800/136272 (1%)
    tile 2000/136272 (1%)
    tile 2200/136272 (2%)
    tile 2400/136272 (2%)
    tile 2600/136272 (2%)
    tile 2800/136272 (2%)
    tile 3000/136272 (2%)
    tile 3200/136272 (2%)
    tile 3400/136272 (2%)
    tile 3600/136272 (3%)
    tile 3800/136272 (3%)
    tile 4000/136272 (3%)
    tile 4200/136272 (3%)
    tile 4400/136272 (3%)
    tile 4600/136272 (3%)
    tile 4800/136272 (4%)
    tile 5000/136272 (4%)
    tile 5200/136272 (4%)
    tile 5400/136272 (4%)
    tile 5600/136272 (4%)
    tile 5800/136272 (4%)
    tile 6000/136272 (4%)
    tile 6200/136272 (5%)
    tile 6400/136272 (5%)
    tile 6600/136272 (5%)
    tile 68

In [ ]:
# ── Cell 7 — Run GrandQC at 7× ───────────────────────────────────────────────
import subprocess, sys, shutil, os, gc, torch
from pathlib import Path

WORK_DIR     = Path('/content/grandqc_idc')
PIPELINE_DIR = WORK_DIR / 'grandqc' / '01_WSI_inference_OPENSLIDE_QC'
TIFF_DIR     = WORK_DIR / 'tiff'
QC_BASE      = WORK_DIR / 'grandqc_output'

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# numpy<2 required for cv2 compatibility in subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy<2', '-q'], check=True)
result = subprocess.run(
    ['python3', '-c', 'import cv2, numpy; print(f"cv2: {cv2.__version__}  numpy: {numpy.__version__}")'],
    capture_output=True, text=True)
print(result.stdout.strip())

env = os.environ.copy()
env['PYTHONNOUSERSITE'] = '1'

def run_stage(script, label, output_dir, extra=[]):
    print(f'\n  ▶ {label}')
    r = subprocess.run(
        [sys.executable, '-W', 'ignore', script,
         '--slide_folder', str(TIFF_DIR),
         '--output_dir',   str(output_dir)] + extra,
        cwd=str(PIPELINE_DIR), capture_output=True, text=True, env=env
    )
    if r.stdout: print(r.stdout[-2000:])
    if r.returncode != 0: print('STDERR:', r.stderr[-1500:])
    else: print(f'  ✓ {label} complete')
    return r.returncode == 0

# Clear previous 7× outputs
qc_dir = QC_BASE / 'qc_mpp15'
if qc_dir.exists():
    shutil.rmtree(qc_dir)
    print('  cleared qc_mpp15')
qc_dir.mkdir(parents=True)

# Stage 1: Tissue detection (shared across magnifications)
print('=' * 60)
print('TISSUE SEGMENTATION')
print('=' * 60)
ok_tis = run_stage('wsi_tis_detect.py', 'Tissue segmentation', qc_dir)

# Stage 2: Artifact segmentation at 7× (MPP 1.5)
print(f'\n{"=" * 60}')
print('ARTIFACT SEGMENTATION — 7× (MPP 1.5)')
print('=' * 60)
ok_art = run_stage('main.py', 'Artifact segmentation 7×', qc_dir,
                   ['--mpp_model', '1.5', '--create_geojson', 'Y'])

# Summary
print(f'\n{"=" * 60}')
print('SUMMARY')
print('=' * 60)
print(f'  {"✓" if ok_tis else "✗"} Tissue segmentation')
print(f'  {"✓" if ok_art else "✗"} Artifact segmentation 7×')
expected = len(list(TIFF_DIR.glob('*.tiff')))
mask_dir = qc_dir / 'mask_qc'
actual   = len(list(mask_dir.glob('*_mask.png'))) if mask_dir.exists() else 0
status   = '✓' if actual == expected else f'⚠️  only {actual}/{expected}'
print(f'  {status} slides processed')


cv2: 4.13.0  numpy: 1.26.4
  cleared qc_mpp15
TISSUE SEGMENTATION

  ▶ Tissue segmentation
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/tf_efficientnet_b0_aa-827b6e33.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientnet_b0_aa-827b6e33.pth

Working with:  breast__tcga_brca__637595675116.2.0.tiff
Overhang (< 1 patch) for width and height:  236 , 115

  ✓ Tissue segmentation complete

ARTIFACT SEGMENTATION — 7× (MPP 1.5)

  ▶ Artifact segmentation 7×

Processing: breast__tcga_brca__637595675116.2.0.tiff

Basic data about processed whole-slide image

Vendor:  generic-tiff
Scan magnification:  99
Number of levels:  9
Level downsamples:  (1.0, 2.0, 4.0, 8.0, 16.001519756838906, 32.00303951367781, 64.00607902735563, 128.19193333561014, 256.7741847081392)
Microns per pixel (slide): 1.0
Height:  42120
Width:  22848
Model patch size at slide MPP:  768 x 768
Width - number of patches:  29
Height - number of patches:  54
Overall number of pat

In [ ]:
# ── Cell 8 — Compute QC metrics → qc_summary_v4.csv ──────────────────────────
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

Image.MAX_IMAGE_PIXELS = None  # disable decompression bomb check for large WSI masks

WORK_DIR = Path('/content/grandqc_idc')
TIFF_DIR = WORK_DIR / 'tiff'
QC_BASE  = WORK_DIR / 'grandqc_output'

labels = {
    1: 'no_artifact', 2: 'fold',      3: 'dark_spot',
    4: 'pen_marking', 5: 'air_bubble', 6: 'out_of_focus'
}

all_results = []
mask_dir = QC_BASE / 'qc_mpp15' / 'mask_qc'

if not mask_dir.exists():
    print('⚠️  mask_qc not found — check Cell 7 ran successfully')
else:
    for mask_file in sorted(mask_dir.glob('*_mask.png')):
        mask = np.array(Image.open(mask_file))
        tiff_name = mask_file.name.replace('_mask.png', '')
        tissue_mask = (mask > 0) & (mask < 7)
        tissue_px = tissue_mask.sum()
        total_px  = mask.size
        row = {
            'tiff_name':           tiff_name,
            'magnification':       '7x',
            'pct_tissue_of_slide': round(tissue_px / total_px * 100, 2),
        }
        if tissue_px > 0:
            for code, name in labels.items():
                row[f'pct_{name}'] = round((mask == code).sum() / tissue_px * 100, 2)
        else:
            for code, name in labels.items():
                row[f'pct_{name}'] = 0.0
        all_results.append(row)

results_df = pd.DataFrame(all_results)
results_df.to_csv(WORK_DIR / 'qc_summary_v4.csv', index=False)
print('✓ qc_summary_v4.csv saved\n')

# Display
display_cols = ['tiff_name', 'pct_tissue_of_slide', 'pct_no_artifact',
                'pct_fold', 'pct_out_of_focus', 'pct_pen_marking', 'pct_air_bubble']
sub = results_df[display_cols].copy()
sub['tiff_name'] = sub['tiff_name'].str[-22:]
expected = len(list(TIFF_DIR.glob('*.tiff')))
n = len(sub)
flag = '' if n == expected else f'  ⚠️  {n}/{expected} slides'
print(f'{"=" * 60}\n  7x (MPP 1.5){flag}\n{"=" * 60}')
print(sub.to_string(index=False))

# Pass/fail verdict
def gqc_verdict(x):
    if x >= 80: return 'PASS'
    if x >= 50: return 'BORDERLINE'
    return 'FAIL'

results_df['verdict'] = results_df['pct_no_artifact'].apply(gqc_verdict)
print(f'\n── Verdicts (PASS≥80% · BORDERLINE 50-80% · FAIL<50%) ──')
for _, r in results_df.iterrows():
    print(f'  {r["tiff_name"][-22:]}  {r["verdict"]:10s}  ({r["pct_no_artifact"]:.1f}% clean)')

# Per-collection summary
results_df['collection'] = results_df['tiff_name'].str.extract(r'__(tcga_\w+|cmb_\w+)__')
if results_df['collection'].notna().any():
    print(f'\n── Per-collection summary ──')
    print(results_df.groupby('collection')[
        ['pct_no_artifact', 'pct_fold', 'pct_out_of_focus']
    ].mean().round(2).to_string())


✓ qc_summary_v4.csv saved

  7x (MPP 1.5)
             tiff_name  pct_tissue_of_slide  pct_no_artifact  pct_fold  pct_out_of_focus  pct_pen_marking  pct_air_bubble
_637595675116.2.0.tiff                33.01             99.9      0.06              0.01              0.0             0.0

── Verdicts (PASS≥80% · BORDERLINE 50-80% · FAIL<50%) ──
  _637595675116.2.0.tiff  PASS        (99.9% clean)

── Per-collection summary ──
            pct_no_artifact  pct_fold  pct_out_of_focus
collection                                             
tcga_brca              99.9      0.06              0.01


In [ ]:
# ── Cell 9a — Install and patch HistoQC (Python 3.12 compatible) ──────────────
# HistoQC was written for Python 3.7/3.8 — requires patches for Colab's Python 3.12
# Key patches:
#   1. dict.__init__(self, {}) — Python 3.12 stricter return value checking
#   2. Magnification override — IDC TIFFs don't have openslide.objective-power tag
#   3. Remove BaseImage.BaseImage from pipeline steps — class can't be called as function
import subprocess, sys, shutil
from pathlib import Path

WORK_DIR    = Path('/content/grandqc_idc')
HISTOQC_DIR = Path('/content/histoqc')
HQC_OUT     = WORK_DIR / 'histoqc_output'
HQC_OUT.mkdir(parents=True, exist_ok=True)

if not HISTOQC_DIR.exists():
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/choosehappy/HistoQC.git',
                    str(HISTOQC_DIR)], check=True)
    print('✓ HistoQC cloned')
else:
    print('✓ HistoQC already present')

for pkg in ['scikit-image', 'scikit-learn', 'scipy', 'matplotlib',
            'openslide-python', 'dill', 'geojson', 'opencv-python-headless']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True)
    if r.returncode != 0: print(f'  [warn] {pkg}')

# Patch BaseImage.py
base_image = HISTOQC_DIR / 'histoqc' / 'BaseImage.py'
content    = base_image.read_text()

if 'dict.__init__(self, {})' not in content:
    content = content.replace('dict.__init__(self)', 'dict.__init__(self, {})')
    print('✓ dict.__init__ patched')
else:
    print('[cached] dict.__init__')

old_mag = '        self["base_mag"] = getMag(self, params)'
new_mag = """        if "magnification" in params:
            self["base_mag"] = float(params["magnification"])
        else:
            self["base_mag"] = getMag(self, params)"""
if old_mag in content:
    content = content.replace(old_mag, new_mag)
    print('✓ Magnification override patched')
else:
    print('[cached] magnification override')

base_image.write_text(content)

for d in HISTOQC_DIR.rglob('__pycache__'):
    shutil.rmtree(d, ignore_errors=True)
print('✓ pycache cleared')

# Config — min_size=50 for sparse biopsy compatibility (vs default 1000)
# Note: BaseImage.BaseImage intentionally excluded from pipeline steps
config_text = """[pipeline]
steps= LightDarkModule.getIntensityThresholdPercent:darktissue
    BlurDetectionModule.identifyBlurryRegions
    BubbleRegionByRegion.detectSmoothness
    BasicModule.finalProcessingSpur
    BasicModule.finalProcessingArea

[BaseImage.BaseImage]
level=0
magnification=40

[LightDarkModule.getIntensityThresholdPercent:darktissue]

[BlurDetectionModule.identifyBlurryRegions]
blur_radius=7
blur_threshold=0.1

[BubbleRegionByRegion.detectSmoothness]

[BasicModule.finalProcessingSpur]
disk_size=9
debug=False

[BasicModule.finalProcessingArea]
min_size=50
debug=False
"""

config_path = HISTOQC_DIR / 'histoqc' / 'config' / 'idc_tiff.ini'
config_path.write_text(config_text)
print(f'✓ Config written (min_size=50)')
print(f'\nReady — {len(list((WORK_DIR/"tiff").glob("*.tiff")))} TIFFs queued for HistoQC')


✓ HistoQC already present
[cached] dict.__init__
✓ Magnification override patched
✓ pycache cleared
✓ Config written (min_size=50)

Ready — 1 TIFFs queued for HistoQC


In [ ]:
# ── Cell 9b — Run HistoQC ─────────────────────────────────────────────────────
import subprocess, sys, shutil
from pathlib import Path

from pathlib import Path
import shutil

HISTOQC_DIR = Path('/content/histoqc')
f = HISTOQC_DIR / 'histoqc' / 'BaseImage.py'
content = f.read_text()

old = '''        if "magnification" in params:
            self["base_mag"] = float(params["magnification"])
        else:
            if "magnification" in params:
            self["base_mag"] = float(params["magnification"])
        else:
            self["base_mag"] = getMag(self, params)'''

new = '''        if "magnification" in params:
            self["base_mag"] = float(params["magnification"])
        else:
            self["base_mag"] = getMag(self, params)'''

content = content.replace(old, new)
f.write_text(content)

for d in HISTOQC_DIR.rglob('__pycache__'):
    shutil.rmtree(d, ignore_errors=True)
print('✓ Fixed')

WORK_DIR    = Path('/content/grandqc_idc')
HISTOQC_DIR = Path('/content/histoqc')
TIFF_DIR    = WORK_DIR / 'tiff'
HQC_OUT     = WORK_DIR / 'histoqc_output'

if HQC_OUT.exists():
    shutil.rmtree(HQC_OUT)
HQC_OUT.mkdir(parents=True)
print('✓ Output dir cleared')

tiff_files = sorted(TIFF_DIR.glob('*.tiff'))
print(f'Running HistoQC on {len(tiff_files)} TIFFs...')

cmd = [sys.executable, '-m', 'histoqc',
       '-c', str(HISTOQC_DIR / 'histoqc' / 'config' / 'idc_tiff.ini'),
       '-o', str(HQC_OUT),
       '-n', '1', '--force'] + [str(t) for t in tiff_files]

result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(HISTOQC_DIR))

if result.stdout: print(result.stdout[-3000:])
if result.returncode != 0: print('STDERR:', result.stderr[-2000:])
else: print('✓ HistoQC complete')

print('\nOutput files:')
for f in sorted(HQC_OUT.rglob('*')):
    if f.is_file() and f.suffix in ['.tsv', '.log']:
        print(f'  {f.name}  ({f.stat().st_size/1e3:.0f} KB)')

err_log = HQC_OUT / 'error.log'
if err_log.exists() and err_log.stat().st_size > 0:
    print('\n── error.log ──')
    print(err_log.read_text())


✓ Fixed
✓ Output dir cleared
Running HistoQC on 1 TIFFs...
✓ HistoQC complete

Output files:
  error.log  (0 KB)
  results.tsv  (1 KB)


In [ ]:
# ── Cell 9c — GrandQC vs HistoQC comparison ──────────────────────────────────
# Verdicts:
#   GrandQC: PASS ≥80% no-artifact · BORDERLINE 50-80% · FAIL <50%
#   HistoQC: PASS <20% blur · FAIL ≥20% blur · UNRELIABLE = tissue removed by cleanup
import pandas as pd
import numpy as np
from pathlib import Path

WORK_DIR = Path('/content/grandqc_idc')
HQC_OUT  = WORK_DIR / 'histoqc_output'

tsv_candidates = list(HQC_OUT.rglob('results.tsv'))
if not tsv_candidates:
    print('⚠️  results.tsv not found — check Cell 9b output')
else:
    with open(tsv_candidates[0]) as f:
        lines = f.readlines()

    header_line = None
    data_lines  = []
    for l in lines:
        if l.startswith('#dataset:'):
            header_line = l.strip().replace('#dataset:', '')
        elif not l.startswith('#') and l.strip():
            data_lines.append(l.strip())

    cols = header_line.split('\t')
    rows = [l.split('\t') for l in data_lines]
    hqc  = pd.DataFrame(rows, columns=cols[:len(rows[0])])
    hqc['slide_id']             = hqc['filename'].str.extract(r'(\d{12,16})\.')[0]
    hqc['blurry_removed_percent'] = pd.to_numeric(hqc['blurry_removed_percent'], errors='coerce')
    hqc['hqc_pct_blurry']       = (1 - hqc['blurry_removed_percent']) * 100
    hqc['hqc_tissue_removed']   = hqc['warnings'].str.contains('NO tissue remains', na=False) if 'warnings' in hqc.columns else False

    gqc = pd.read_csv(WORK_DIR / 'qc_summary_v4.csv')
    gqc['slide_id']   = gqc['tiff_name'].str.extract(r'(\d{12,16})\.')[0]
    gqc['collection'] = gqc['tiff_name'].str.extract(r'__(tcga_\w+|cmb_\w+)__')

    def gqc_verdict(x):
        if x >= 80: return 'PASS'
        if x >= 50: return 'BORDERLINE'
        return 'FAIL'

    def hqc_verdict(row):
        if row['hqc_tissue_removed']: return 'UNRELIABLE'
        if row['hqc_pct_blurry'] > 20: return 'FAIL'
        return 'PASS'

    gqc['gqc_verdict'] = gqc['pct_no_artifact'].apply(gqc_verdict)
    hqc['hqc_verdict'] = hqc.apply(hqc_verdict, axis=1)

    merged = gqc.merge(
        hqc[['slide_id','hqc_pct_blurry','hqc_tissue_removed','hqc_verdict']],
        on='slide_id', how='left'
    )

    print('=' * 90)
    print('GrandQC (7x) vs HistoQC — Comparison')
    print('=' * 90)
    print(f'{"Collection":<12} {"Slide":<18} {"GQC No-Art%":>11} {"GQC":>12} {"HQC Blur%":>10} {"HQC":>12}')
    print('-' * 90)
    for _, r in merged.iterrows():
        blur  = f"{r['hqc_pct_blurry']:.1f}%" if pd.notna(r.get('hqc_pct_blurry')) else 'N/A'
        hqc_v = str(r.get('hqc_verdict', 'N/A'))
        coll  = str(r.get('collection', '?'))
        sid   = str(r.get('slide_id', '?'))[-12:]
        print(f'{coll:<12} {sid:<18} {r["pct_no_artifact"]:>11.2f} {r["gqc_verdict"]:>12} {blur:>10} {hqc_v:>12}')

    valid = merged[merged.get('hqc_verdict', pd.Series(dtype=str)) != 'UNRELIABLE']
    agree = (valid['gqc_verdict'].map({'PASS':'PASS','BORDERLINE':'FAIL','FAIL':'FAIL'})
             == valid.get('hqc_verdict', pd.Series(dtype=str))).sum() if len(valid) > 0 else 0
    print(f'\nValid comparisons: {len(valid)}/{len(merged)}')
    print(f'Agreement: {agree}/{max(len(valid),1)} ({agree/max(len(valid),1)*100:.0f}%)')
    print('\nNote: UNRELIABLE = HistoQC finalProcessingSpur removed all tissue')
    print('      (biopsy cores < min_size threshold; inherent HistoQC limitation)')

    merged.to_csv(WORK_DIR / 'grandqc_histoqc_comparison.csv', index=False)
    print('\n✓ Saved: grandqc_histoqc_comparison.csv')


GrandQC (7x) vs HistoQC — Comparison
Collection   Slide              GQC No-Art%          GQC  HQC Blur%          HQC
------------------------------------------------------------------------------------------
tcga_brca    637595675116             99.90         PASS      22.3%         FAIL

Valid comparisons: 1/1
Agreement: 0/1 (0%)

Note: UNRELIABLE = HistoQC finalProcessingSpur removed all tissue
      (biopsy cores < min_size threshold; inherent HistoQC limitation)

✓ Saved: grandqc_histoqc_comparison.csv


In [ ]:
# ── Cell 10 — Clean GeoJSONs + package QuPath zip — TCGA_BRCA_0.5MPP ────────
import json, zipfile, subprocess, sys, shutil
from pathlib import Path

WORK_DIR = Path('/content/grandqc_idc')
TIFF_DIR = WORK_DIR / 'tiff'
QC_BASE  = WORK_DIR / 'grandqc_output'

geojson_dir = QC_BASE / 'qc_mpp15' / 'geojson_qc'

# Install shapely for geometry validation
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shapely'], check=True)

# Validate
if geojson_dir.exists():
    files = sorted(geojson_dir.glob('*.geojson'))
    total = sum(len(json.load(open(f))['features']) for f in files)
    print(f'GeoJSON inventory: {len(files)} slides, {total} features')
else:
    print('⚠️  geojson_qc not found')

# Clean GeoJSONs with Shapely — buffer(0) + round coordinates for JTS compatibility
print('\nCleaning GeoJSONs...')
if geojson_dir.exists():
    from shapely.geometry import shape, mapping
    from shapely.validation import make_valid

    for gj_path in sorted(geojson_dir.glob('*.geojson')):
        with open(gj_path) as f:
            gj = json.load(f)

        original = len(gj['features'])
        valid_features = []
        dropped = 0

        for feat in gj['features']:
            try:
                geom = shape(feat['geometry'])
                if not geom.is_valid:
                    geom = make_valid(geom)
                geom = geom.buffer(0)
                if geom.is_empty or geom.area <= 1.0:
                    dropped += 1
                    continue
                if geom.geom_type == 'MultiPolygon':
                    geom = max(geom.geoms, key=lambda g: g.area)
                geo_dict = mapping(geom)
                def round_coords(coords):
                    return [tuple(round(c, 1) for c in pt) for pt in coords]
                if geo_dict['type'] == 'Polygon':
                    geo_dict['coordinates'] = [round_coords(ring) for ring in geo_dict['coordinates']]
                feat['geometry'] = geo_dict
                valid_features.append(feat)
            except Exception:
                dropped += 1

        gj['features'] = valid_features
        with open(gj_path, 'w') as f:
            json.dump(gj, f)
        print(f'  {gj_path.name[-40:]}: {original} → {len(valid_features)} features ({dropped} dropped)')

# Rename CSVs for this collection
for old_name, new_name in [
    ('qc_summary_v4.csv', 'TCGA_BRCA_0.5MPP_qc_summary.csv'),
    ('grandqc_histoqc_comparison.csv', 'TCGA_BRCA_0.5MPP_grandqc_histoqc_comparison.csv'),
]:
    src = WORK_DIR / old_name
    dst = WORK_DIR / new_name
    if src.exists():
        shutil.copy2(src, dst)
        print(f'✓ {old_name} → {new_name}')

# QuPath Groovy script — bypasses PathIO.readObjects to avoid JTS errors,
# creates rectangle ROIs directly from GeoJSON coordinates
groovy = '''
// ── CONFIG ────────────────────────────────────────────────────────────────────
def GEOJSON_ROOT = "REPLACE_WITH_YOUR_PATH"
// e.g. "C:/Users/ronak/Desktop/TCGA_BRCA_0.5MPP_grandqc_qupath/geojsons"
// Use forward slashes — backslashes cause parse errors

def classColors = [
    "No Artifact"              : ColorTools.makeRGB(0,   200, 0),
    "Fold"                     : ColorTools.makeRGB(255, 140, 0),
    "Dark Spot"                : ColorTools.makeRGB(80,  80,  80),
    "Pen Marking"              : ColorTools.makeRGB(255, 0,   255),
    "Air Bubble"               : ColorTools.makeRGB(0,   180, 255),
    "OOF"                      : ColorTools.makeRGB(255, 0,   0),
    "Darkspot & Foreign Object": ColorTools.makeRGB(80,  80,  80),
]

def imageData = getCurrentImageData()
def imageName = imageData.getServer().getMetadata().getName()
def geojsonFile = new File(GEOJSON_ROOT + "/7x/" + imageName + ".geojson")

print "Image: " + imageName
print "GeoJSON exists: " + geojsonFile.exists()
if (!geojsonFile.exists()) { print "ERROR: not found"; return }

import com.google.gson.JsonParser
import qupath.lib.objects.PathObjects
import qupath.lib.roi.ROIs
import qupath.lib.regions.ImagePlane

def plane = ImagePlane.getDefaultPlane()
def root = JsonParser.parseString(geojsonFile.text).getAsJsonObject()
def features = root.getAsJsonArray("features")

def loaded = 0
def skipped = 0

features.each { feature ->
    try {
        def props = feature.getAsJsonObject("properties")
        def label = props.has("classification") ?
            props.get("classification").getAsString() : "Unknown"

        def geom = feature.getAsJsonObject("geometry")
        def coords = geom.getAsJsonArray("coordinates").get(0).getAsJsonArray()

        def xs = []
        def ys = []
        coords.each { pt ->
            xs.add(pt.get(0).getAsDouble())
            ys.add(pt.get(1).getAsDouble())
        }

        def x = xs.min()
        def y = ys.min()
        def w = xs.max() - x
        def h = ys.max() - y

        if (w < 2 || h < 2) { skipped++; return }

        def roi = ROIs.createRectangleROI(x, y, w, h, plane)
        def color = classColors[label] ?: ColorTools.makeRGB(128,128,128)
        def pathClass = PathClass.fromString(label, color)
        def annotation = PathObjects.createAnnotationObject(roi, pathClass)
        addObject(annotation)
        loaded++
    } catch (Exception e) {
        skipped++
    }
}

fireHierarchyUpdate()

def entry = getProjectEntry()
if (entry != null) {
    entry.saveImageData(imageData)
    getProject().syncChanges()
}
print "Loaded " + loaded + " annotations, skipped " + skipped
print "Restart QuPath to see saved annotations"
'''

readme = """GrandQC QuPath Annotation Loader — TCGA_BRCA_0.5MPP
=====================================================
1. Extract zip to local folder
2. Open QuPath, create project, add TIFFs from tiff/ folder
3. Open Automate > Script Editor
4. Paste TCGA_BRCA_0.5MPP_grandqc_load_annotations.groovy
5. Set GEOJSON_ROOT to your local geojsons path (forward slashes)
6. Open a slide and click Run
7. RESTART QuPath to see saved annotations

Colors: No Artifact=green  Fold=orange  OOF=red
        Air Bubble=cyan  Pen Marking=magenta  Dark Spot=gray

Magnification: 7x (MPP 1.5) — GrandQC paper benchmark
Resolution: 0.5 MPP (downsampled from native 0.25 MPP)
Pass/fail: PASS>=80%  BORDERLINE 50-80%  FAIL<50% no-artifact
"""

script_path = WORK_DIR / 'TCGA_BRCA_0.5MPP_grandqc_load_annotations.groovy'
readme_path = WORK_DIR / 'TCGA_BRCA_0.5MPP_README_QuPath.txt'
script_path.write_text(groovy.strip())
readme_path.write_text(readme)
print('\n✓ Groovy script written')

# Build zip
zip_path = WORK_DIR / 'TCGA_BRCA_0.5MPP_grandqc_qupath_package.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(script_path, 'TCGA_BRCA_0.5MPP_grandqc_qupath/TCGA_BRCA_0.5MPP_grandqc_load_annotations.groovy')
    zf.write(readme_path, 'TCGA_BRCA_0.5MPP_grandqc_qupath/TCGA_BRCA_0.5MPP_README_QuPath.txt')

    for csv_name in ['TCGA_BRCA_0.5MPP_qc_summary.csv', 'TCGA_BRCA_0.5MPP_grandqc_histoqc_comparison.csv']:
        p = WORK_DIR / csv_name
        if p.exists():
            zf.write(p, f'TCGA_BRCA_0.5MPP_grandqc_qupath/{csv_name}')

    if geojson_dir.exists():
        for gj in sorted(geojson_dir.glob('*.geojson')):
            zf.write(gj, f'TCGA_BRCA_0.5MPP_grandqc_qupath/geojsons/7x/{gj.name}')

    for tiff in sorted(TIFF_DIR.glob('*.tiff')):
        print(f'  adding {tiff.name} ({tiff.stat().st_size/1e6:.0f} MB)')
        zf.write(tiff, f'TCGA_BRCA_0.5MPP_grandqc_qupath/tiff/{tiff.name}')

size_mb = zip_path.stat().st_size / 1e6
print(f'\n✓ Package: TCGA_BRCA_0.5MPP_grandqc_qupath_package.zip ({size_mb:.0f} MB)')

from google.colab import files
files.download(str(zip_path))

GeoJSON inventory: 1 slides, 190 features

Cleaning GeoJSONs...
  tcga_brca__637595675116.2.0.tiff.geojson: 190 → 190 features (0 dropped)
✓ qc_summary_v4.csv → TCGA_BRCA_0.5MPP_qc_summary.csv
✓ grandqc_histoqc_comparison.csv → TCGA_BRCA_0.5MPP_grandqc_histoqc_comparison.csv

✓ Groovy script written
  adding breast__tcga_brca__637595675116.2.0.tiff (150 MB)

✓ Package: TCGA_BRCA_0.5MPP_grandqc_qupath_package.zip (137 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>